In [1]:
import importlib
import NHL_script
importlib.reload(NHL_script)

# NHL DATA SCRAPE FROM MONEYPUCK
NHL_script.get_nhl_skaters()
NHL_script.get_nhl_goalies()
NHL_script.get_nhl_lines()
NHL_script.get_nhl_teams()

# Process Data
# NHL_data.process_nhl_data_and_generate_html()
NHL_script.combine_and_save_skaters(2, 'NHL_data/SOG_per_game.csv')
NHL_script.csv_to_html('NHL_data/SOG_per_game.csv')
#NHL_script.add_checkboxes_to_html('NHL_data/SOG_per_game.html')
#NHL_script.rename_csv_headers()
NHL_script.make_nhl_report_today()

File 'nhl_skaters_2025_20260119.csv' already exists. No action needed.
File 'nhl_goalies_2025_20260119.csv' already exists. No action needed.
File 'nhl_lines_2025_20260119.csv' already exists. No action needed.
File 'nhl_teams_2025_20260119.csv' already exists. No action needed.


/home/codespace/.local/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:4268: RuntimeWarning: Degrees of freedom <= 0 for slice
  return _methods._var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/codespace/.local/lib/python3.12/site-packages/numpy/_core/_methods.py:181: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/home/codespace/.local/lib/python3.12/site-packages/numpy/_core/_methods.py:215: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


name Kevin Fiala
team LAK
pos L
gp 48
eG25 15.69
aG24 0.44
aG25 0.35
24-25 18.0
a24-25 0.53
G22 23
G23 29
G24 35
G25 17.0
pastG 0-0-1-0-0-0-1-1-0-1-0-0-1-0
Gvar 0.23
aP 0.71
P 34
pastP 1-0-2-0-0-0-2-1-1-2-1-0-1-0
Pvar 0.6
eSOG 155
aSOG 2.85
SOG 137.0
pastSOG 4-4-3-0-1-4-4-3-4-3-3-1-2-1
SOGvar 1.8
Gpick  
Ppick  
Spick  
name Alex Tuch
team BUF
pos R
gp 47
eG25 12.9
aG24 0.45
aG25 0.36
24-25 19.0
a24-25 0.54
G22 36
G23 22
G24 36
G25 17.0
pastG 0-1-1-0-1-0-1-1-0-0-1-0-0-0
Gvar 0.24
aP 0.85
P 40
pastP 0-1-1-1-1-1-1-2-0-1-1-0-1-0
Pvar 0.31
eSOG 142
aSOG 2.38
SOG 112.0
pastSOG 0-2-1-0-2-2-1-3-5-2-2-2-3-4
SOGvar 1.78
Gpick  
Ppick  
Spick  
name Zach Hyman
team EDM
pos L
gp 31
eG25 20.44
aG24 0.34
aG25 0.61
24-25 8.0
a24-25 0.16
G22 36
G23 54
G24 27
G25 19.0
pastG 3-0-2-1-0-1-0-0-1-1-0-1
Gvar 0.81
aP 1.0
P 31
pastP 4-0-2-2-0-1-0-0-1-1-1-3
Pvar 1.52
eSOG 109
aSOG 3.03
SOG 94.0
pastSOG 7-4-3-3-4-2-4-3-4-2-6-6
SOGvar 2.33
Gpick  
Ppick  
Spick  
name Quinton Byfield
team LAK
pos R
gp 47
eG25 12

In [11]:
import csv
import requests
from bs4 import BeautifulSoup
import requests

def save_url_to_text_file(url, output_file_path):
    """
    Fetches the content of a URL and saves it to a text file.

    Args:
        url (str): The URL to fetch.
        output_file_path (str): The path to save the content as a text file.

    Returns:
        None
    """
    try:
        # Fetch the webpage
        response = requests.get(url)
        response.raise_for_status()  # Raise an error for bad status codes

        # Save the content to a text file
        with open(output_file_path, 'w', encoding='utf-8') as file:
            file.write(response.text)

        print(f"Content saved to {output_file_path}")
    except Exception as e:
        print(f"Error fetching or saving the URL content: {e}")
def get_all_games_from_url(url):
    """
    Fetches the 'ALL GAMES' section from the given URL and parses the game data.

    Args:
        url (str): The URL of the page to scrape.

    Returns:
        list: A list of game data, where each game is represented as a string.
    """
    try:
        # Fetch the webpage
        response = requests.get(url)
        response.raise_for_status()  # Raise an error for bad status codes

        # Parse the HTML content
        soup = BeautifulSoup(response.text, 'html.parser')

        # Find the "ALL GAMES" section
        all_games_section = soup.find(string="ALL GAMES:").find_next("pre")
        if not all_games_section:
            raise ValueError("Could not find the 'ALL GAMES' section on the page.")

        # Extract the text and split into lines
        all_games_text = all_games_section.get_text()
        all_games_lines = all_games_text.splitlines()

        # Remove empty lines and return the data
        return [line.strip() for line in all_games_lines if line.strip()]

    except Exception as e:
        print(f"Error fetching or parsing the URL: {e}")
        return []


def generate_matchup_urls(matchups):
    """
    Generates URLs for each matchup in both directions.

    Args:
        matchups (list): A list of matchups, where each matchup is a list of two team abbreviations.

    Returns:
        list: A list of URLs for each matchup in both directions.
    """
    urls = []
    for team1, team2 in matchups:
        urls.append(f"https://mcubed.net/nhl/{team1}/{team2}.shtml")
        urls.append(f"https://mcubed.net/nhl/{team2}/{team1}.shtml")
    return urls

def replace_team_names_with_other_short(matchups, team_names):
    """
    Replaces team names in matchups with their corresponding 'OTHER_SHORT' values.

    Args:
        matchups (list): A list of matchups, where each matchup is a list of two team names.
        team_names (list): A list of team data, where each entry contains team details.

    Returns:
        list: A new list of matchups with team names replaced by 'OTHER_SHORT' values.
    """
    # Create a mapping of team names to their 'OTHER_SHORT' values
    name_to_other_short = {team[1]: team[3] for team in team_names}

    # Replace team names in matchups with their 'OTHER_SHORT' values
    updated_matchups = []
    for matchup in matchups:
        team1, team2 = matchup
        updated_matchups.append([
            name_to_other_short.get(team1, team1),  # Replace team1 if found, else keep original
            name_to_other_short.get(team2, team2)   # Replace team2 if found, else keep original
        ])

    return updated_matchups

def parse_schedule_to_matchups(schedule_text):
    """
    Parses a schedule text into an array of team matchups.

    Args:
        schedule_text (str): The multiline schedule text.

    Returns:
        list: A list of matchups, where each matchup is a list of two teams.
    """
    lines = schedule_text.splitlines()
    matchups = []

    for line in lines:
        # Split the line into parts and extract the teams
        parts = line.split(" - ")
        if len(parts) == 2:
            teams = parts[1].split(" @ ")
            if len(teams) == 2:
                matchups.append([teams[0].strip(), teams[1].strip()])

    return matchups

def text_to_csv(array1, output_csv_path):
    """
    Converts a multiline text string into a CSV file.

    Args:
        text (str): The multiline text string to convert.
        output_csv_path (str): The path to save the CSV file.

    Returns:
        None
    """
    lines = array1
    if not lines:
        raise ValueError("The input text is empty.")

    # Extract headers and data
    headers = lines[0].split()
    data = []

    for line in lines[1:]:
        # Split the line into columns based on whitespace
        # Use rsplit to handle team names with spaces
        parts = line.rsplit(maxsplit=len(headers) - 1)
        data.append(parts)

    # Write to CSV
    with open(output_csv_path, mode='w', newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(headers)  # Write headers
        writer.writerows(data)    # Write data rows

    print(f"CSV file saved to {output_csv_path}")

def split_text_into_lines(multiline_text):
    """
    Splits a multiline text string into an array of lines.

    Args:
        multiline_text (str): The multiline text string to split.

    Returns:
        list: A list of lines from the input text.
    """
    return multiline_text.splitlines()

def remove_indices_from_list(input_list, indices_to_remove):
    """
    Removes multiple entries from a list at specific index locations.

    Args:
        input_list (list): The original list.
        indices_to_remove (list): A list of indices to remove.

    Returns:
        list: A new list with the specified indices removed.
    """
    indices_to_remove = set(indices_to_remove)  # Convert to set for faster lookup
    return [item for idx, item in enumerate(input_list) if idx not in indices_to_remove]

import importlib
import NHL_script
import file_operations
importlib.reload(NHL_script)
importlib.reload(file_operations)
#NHL_schedule_2026-01-20.txt
teams_csv_path = 'NHL_data/static_data/nhl_team_names3.csv'

# standings
beans = NHL_script.get_nhl_standings_now()

# print(beans)
report = NHL_script.generate_text_report(beans["standings"])
report_array = split_text_into_lines(report)


csv_headers_text = report_array[2]
csv_headers_array = [csv_headers_text]
new_report_array = remove_indices_from_list(report_array, [0,1,2,11,12,13,22,23,24,25,34,35,36])
data_array = csv_headers_array + new_report_array
text_to_csv(data_array, 'NHL_data/nhl_standings_today.csv')




# open nhl_team_names3.csv
team_names_csv = file_operations.read_csv(teams_csv_path)
# for x in team_names_csv:
#     print(x)

# open schedule for the day
schedule_path = 'NHL_data/schedule/NHL_schedule_2026-01-20.txt'
schedule_text = file_operations.read_text_file(schedule_path)
# print(schedule_text)
schedule_matchups = parse_schedule_to_matchups(schedule_text)
# for x in schedule_matchups:
#     print(x)

short_match_ups = replace_team_names_with_other_short(schedule_matchups, team_names_csv)
# for x in short_match_ups:
#     print(x)


array2 = generate_matchup_urls(short_match_ups)
# for url in array2:
#     print(url)

# url = "https://mcubed.net/nhl/otw/clb.shtml"
# all_games = get_all_games_from_url(url)

# for game in all_games:
#     print(game)

url = "https://mcubed.net/nhl/otw/clb.shtml"
output_file = "nhl_otw_clb.txt"

save_url_to_text_file(url, output_file)
# get the team matches for the day
# for matchup in schedule_matchups:
#     team1, team2 = matchup
#     link1 = f"https://mcubed.net/nhl/{team1}/{team2}.shtml"
#     link2 = f"https://mcubed.net/nhl/{team2}/{team1}.shtml"
#     print(link1)
#     print(link2)
# parse the html for link and get all games history
# open standings.txt and parse
# make report


CSV file saved to NHL_data/nhl_standings_today.csv
Content saved to nhl_otw_clb.txt
